In [16]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_csv("data/case1/train1.csv")
df["kilo_rpm"] = df["speed_rpm"] / 1000
df["delta_p"] = df["outlet_pressure_bar"] - df["inlet_pressure_bar"]
dataset = df[["power_w", "kilo_rpm", "delta_p", "flow_m3_h"]].copy()

In [ ]:
target = "flow_m3_h"
feature_names = [column for column in dataset.columns if column != target]
X = dataset[feature_names].to_numpy(dtype=float)
y = dataset[target].to_numpy(dtype=float)

In [38]:
test_df = pd.read_csv("data/case1/test1.csv")
test_df["kilo_rpm"] = test_df["speed_rpm"] / 1000
test_df["delta_p"] = test_df["outlet_pressure_bar"] - test_df["inlet_pressure_bar"]
test_dataset = test_df[feature_names + [target]]
X_test = test_dataset[feature_names]
y_test = test_dataset[target]

In [19]:
def mse(values):
    return np.mean((values - np.mean(values)) ** 2)

In [25]:
def find_best_split(X, y, feature_names):
    parent_mse = mse(y)
    best_split = None

    for i in range(len(feature_names)):
        values = np.sort(np.unique(X[:, i]))
        candidate_thresholds = (values[:-1] + values[1:]) / 2

        for threshold in candidate_thresholds:
            left_mask = X[:, i] < threshold
            left_y = y[left_mask]
            right_y = y[~left_mask]

            weighted_mse = (
                len(left_y) * mse(left_y) + len(right_y) * mse(right_y)
            ) / len(y)

            if best_split is None or weighted_mse < best_split["weighted_mse"]:
                best_split = {
                    "feature": feature_names[i],
                    "threshold": threshold,
                    "weighted_mse": weighted_mse,
                    "parent_mse": parent_mse,
                    "mse_reduction": parent_mse - weighted_mse,
                    "left_count": len(left_y),
                    "right_count": len(right_y),
                    "left_prediction": np.mean(left_y),
                    "right_prediction": np.mean(right_y),
                }

    return best_split

In [21]:
def make_leaf(y):
    return {
        "leaf": True,
        "prediction": float(np.mean(y)),
        "n_samples": len(y),
        "mse": float(mse(y)),
    }

In [ ]:
def build_tree(X, y, feature_names, max_depth=3):
    if max_depth < 1 or len(y) < 2:
        return make_leaf(y)

    split = find_best_split(X, y, feature_names)

    if split is None or split["mse_reduction"] <= 0.05:
        return make_leaf(y)

    axis = feature_names.index(split["feature"])
    left_mask = X[:, axis] < split["threshold"]

    return {
        "leaf": False,
        "feature": split["feature"],
        "axis": axis,
        "threshold": split["threshold"],
        "n_samples": len(y),
        "mse_reduction": split["mse_reduction"],
        "left": build_tree(
            X[left_mask], y[left_mask], feature_names, max_depth -1
        ),
        "right": build_tree(
            X[~left_mask], y[~left_mask], feature_names, max_depth -1
        ),
    }


tree = build_tree(X, y, feature_names, max_depth=5)

In [33]:
def print_tree(node, spacing=""):
    if node["leaf"]:
        print(
            f"{spacing}Predict flow_m3_h = {node['prediction']:.2f} "
            f"(samples: {node['n_samples']}; MSE: {node['mse']:.3f})"
        )
        return

    print(
        f"{spacing}{node['feature']} < {node['threshold']:.2f}? "
        f"(samples: {node['n_samples']})"
    )

    print(f"{spacing}--> True:")
    print_tree(node["left"], spacing + "  ")

    print(f"{spacing}--> False:")
    print_tree(node["right"], spacing + "  ")


def total_leaf_squared_error(node):
    if node["leaf"]:
        return node["mse"] * node["n_samples"]

    return (
        total_leaf_squared_error(node["left"])
        + total_leaf_squared_error(node["right"])
    )


starting_mse = mse(y)
final_mse = total_leaf_squared_error(tree) / len(y)
print(f"Starting MSE (single mean prediction): {starting_mse:.3f}")
print(f"Final tree training MSE: {final_mse:.3f}")
print(f"MSE reduction: {starting_mse - final_mse:.3f}\n")
print_tree(tree)

Starting MSE (single mean prediction): 170.382
Final tree training MSE: 15.995
MSE reduction: 154.387

power_w < 1085.19? (samples: 480)
--> True:
  power_w < 523.12? (samples: 368)
  --> True:
    delta_p < 0.14? (samples: 232)
    --> True:
      kilo_rpm < 2.07? (samples: 33)
      --> True:
        kilo_rpm < 1.64? (samples: 24)
        --> True:
          Predict flow_m3_h = 14.13 (samples: 9; MSE: 5.029)
        --> False:
          Predict flow_m3_h = 18.96 (samples: 15; MSE: 3.971)
      --> False:
        power_w < 278.90? (samples: 9)
        --> True:
          Predict flow_m3_h = 25.34 (samples: 5; MSE: 0.794)
        --> False:
          Predict flow_m3_h = 31.79 (samples: 4; MSE: 4.294)
    --> False:
      power_w < 180.85? (samples: 199)
      --> True:
        delta_p < 0.19? (samples: 63)
        --> True:
          Predict flow_m3_h = 8.73 (samples: 17; MSE: 18.557)
        --> False:
          Predict flow_m3_h = 4.50 (samples: 46; MSE: 8.095)
      --> False:
     

In [34]:
def predict_one(node, row):
    while not node["leaf"]:
        if row[node["axis"]] < node["threshold"]:
            node = node["left"]
        else:
            node = node["right"]

    return node["prediction"]

In [40]:
X_test_array = X_test.to_numpy(dtype=float)
predictions = np.array([predict_one(tree, row) for row in X_test_array])
test_mse = np.mean((y_test - predictions) ** 2)
test_mse

np.float64(28.66663268991185)